# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://senscience.ai/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset covers mental health survey responses, demographics, and standardized assessment scores (GAD-7, PHQ-9, PSQ) collected from residents of Kilifi County, Kenya.

### Dataset Source
The dataset is described using a Croissant schema accessible at:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Explore available `RecordSet`s and `Field`s by their `@id`s.

We'll list the record sets, the field IDs, their names, and associated columns. All references use the Croissant `@id` pattern as required.

In [ ]:
# List all RecordSets and their Fields with @id references
record_sets = list(metadata.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet: @id={rs.id}, name={rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for f in rs.fields:
        col_ids = [col.id for col in f.columns] if getattr(f, 'columns', None) else []
        print(f"    Field: @id={f.id}, name={f.name}, dataType={f.data_type}, columns={col_ids}")
    print()

To view a few records from any record set, use the `@id` of the desired record set as shown below.

In [ ]:
# Display a few records from the main RecordSet (update the ID as appropriate)
# Here we assume there's a main record set with demographic and survey data.
main_record_set_id = None
for rs in record_sets:
    if 'demographics' in rs.name.lower() or 'main' in rs.name.lower() or 'survey' in rs.name.lower():
        main_record_set_id = rs.id
        break
# If not found, use the first record set
if main_record_set_id is None and record_sets:
    main_record_set_id = record_sets[0].id

print(f"Showing 2 sample records from RecordSet @id={main_record_set_id}\n")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(rec)
    if i >= 1: break

## 3. Data Extraction
Extract all tabular record sets into pandas DataFrames for analysis. We'll use their `@id` as the DataFrame keys for referencing.

In [ ]:
# Extract data from each record set to DataFrames
dataframes = {}
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        if records:  # Only store non-empty tables
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            print(f"Loaded DataFrame for RecordSet @id={rs.id}, shape={df.shape}")
    except Exception as e:
        print(f"Could not load record set {rs.id}: {e}")

# Show the columns of the main record set
print(f"\nColumns in DataFrame for main RecordSet (@id={main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's analyze key numeric fields (e.g., GAD-7, PHQ-9, PSQ scores). All field references use their Croissant `@id` as per the dataset schema.

You can adjust the `numeric_field_id` and `group_field_id` below according to record set and field overviews.

In [ ]:
# Example field IDs (substitute as appropriate based on the overview section!)
# For illustration, we guess field IDs for GAD-7, PHQ-9 etc. based on typical schemas
gad7_field_id = None
phq9_field_id = None
psq_field_id = None
age_field_id = None
group_field_id = None

for rs in record_sets:
    if rs.id == main_record_set_id:
        for f in rs.fields:
            fid = f.id.lower()
            if 'gad7' in fid or 'gad-7' in fid:
                gad7_field_id = f.id
            if 'phq9' in fid or 'phq-9' in fid:
                phq9_field_id = f.id
            if 'psq' in fid:
                psq_field_id = f.id
            if 'age' in fid:
                age_field_id = f.id
            if group_field_id is None and (fid.endswith('gender') or fid.endswith('sex')):
                group_field_id = f.id

print("Chosen field IDs:")
print(f"  GAD-7 @id: {gad7_field_id}")
print(f"  PHQ-9 @id: {phq9_field_id}")
print(f"  PSQ @id: {psq_field_id}")
print(f"  Age @id: {age_field_id}")
print(f"  Grouping field @id: {group_field_id}\n")

# Use the first available numeric field for analysis, default to GAD-7 if present
numeric_field_id = gad7_field_id or phq9_field_id or psq_field_id or age_field_id or dataframes[main_record_set_id].select_dtypes('number').columns[0]

df = dataframes[main_record_set_id]

# Remove non-numeric or missing values and filter for large values (> threshold)
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in '@id={main_record_set_id}' with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Add normalized column
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    # Group by categorical field if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count','mean','std'])
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in @id={main_record_set_id}; please adjust field IDs as needed.")

## 5. Visualization
Let's visualize the distribution of the main numeric field (`@id` as above), and (if available) show the average by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print(f"Field {numeric_field_id} not in DataFrame for visualization.")

## 6. Conclusion

* We loaded the FAIR² Kilifi County mental health survey data via `mlcroissant` using only Croissant `@id` identifiers.
* We examined schema structure (record sets, fields, and columns) programmatically by their `@id`s.
* After extracting data to DataFrames (keyed by `@id`), we filtered, normalized, grouped, and visualized numeric survey scores.

**Next steps:**
- Explore distributions and relationships for additional fields/record sets using their `@id`s.
- Refine filtering, normalization, and grouping for new analyses as needed.
- Consider exporting cleaned/derived DataFrames for downstream ML or reporting workflows.